# SMEsD round 4 (timing-free, tree-like inputs, richer aggregation, ranking loss) on Kaggle

Runs the ten pre-declared variants in `benchmark/run4.py` (all without the lawsuit timing columns), 10 seeds each, CPU only, four at a time:

| Variant | What it adds to the timing-free reference `T0` |
|---|---|
| `T2` | two message-passing layers (control for `TA`) |
| `TP` | two-hop context and the sector-conditioned blend |
| `TL` | lawsuit vectors without age inputs |
| `TE` | quantile (piecewise-linear) numeric embedding |
| `TA` | `T2` + sum/max/mean aggregation, degree MLP, jumping knowledge |
| `TR` | pairwise ranking loss |
| `TEAR` | embedding + aggregators + ranking loss |
| `TLEAR` | `TEAR` + lawsuit vectors |
| `T0NA` | diagnostic: also drop set-up age |

**Setup**: create a *private* dataset from `smesd_round4_kaggle.zip` (made by `scripts/make_kaggle_zip.sh`) and add it to this notebook. Accelerator: **None**. Then **Save Version -> Save & Run All (Commit)** so it keeps running without your browser. Expect roughly 1.5 to 3 hours.
**Result**: download `round4_results.zip` from the notebook Output tab and send it back (or paste the printed report).

Validation picks the headline; the per-seed log prints validation AUC only.

Note: the SMEsD repository has no license file; keep the dataset private and cite Wei et al., Information Sciences 2024.

In [ ]:
SEEDS = 10            # lower to 5 for a faster run
THREADS = 1           # per process
WORKERS = 4           # processes in parallel (Kaggle CPU has 4 cores)
# Slowest first, so the pool stays busy to the end.
VARIANTS = ["TLEAR", "TL", "TEAR", "TP", "TA", "TE", "TR", "T2", "T0", "T0NA"]
EXTRA_ARGS = []       # e.g. ["--epochs", "2"] for a quick smoke test

In [ ]:
import glob, os, shutil, subprocess, sys, time, zipfile
WORK = "/kaggle/working/smesd_pivot"
hits = glob.glob("/kaggle/input/**/benchmark/run4.py", recursive=True)
if hits:                                   # Kaggle already extracted the zip
    src = os.path.dirname(os.path.dirname(hits[0]))
    shutil.copytree(src, WORK, dirs_exist_ok=True)
else:
    z = glob.glob("/kaggle/input/**/smesd_round4_kaggle.zip", recursive=True)
    assert len(z) == 1, f"attach the dataset first, found {z}"
    zipfile.ZipFile(z[0]).extractall("/kaggle/working")
os.makedirs(WORK + "/results", exist_ok=True)
open(WORK + "/benchmark/__init__.py", "a").close()      # regular package, so an installed 'benchmark' cannot shadow it
os.environ["PYTHONPATH"] = WORK
sys.path.insert(0, WORK)
import benchmark, risk_gnn
print("benchmark imported from:", benchmark.__file__)
assert benchmark.__file__.startswith(WORK), "wrong 'benchmark' package"
print(sorted(os.listdir(WORK)))

In [ ]:
def launch(v):
    cmd = [sys.executable, "-W", "ignore", "-m", "benchmark.run4", "--variant", v, "--tag", "r4_" + v,
           "--dropout", "0.1", "--wd", "1e-4", "--seeds", str(SEEDS), "--threads", str(THREADS), *EXTRA_ARGS]
    return subprocess.Popen(cmd, cwd=WORK, stdout=open(f"{WORK}/results/log_r4_{v}.txt", "w"), stderr=subprocess.STDOUT)

def progress(v):
    return sum(" seed " in l for l in open(f"{WORK}/results/log_r4_{v}.txt"))

queue, running, codes, start = list(VARIANTS), {}, {}, time.time()
while queue or running:
    while queue and len(running) < WORKERS:
        v = queue.pop(0); running[v] = launch(v)
    time.sleep(60)
    for v in [v for v, p in running.items() if p.poll() is not None]:
        codes[v] = running.pop(v).returncode
    print(f"{(time.time()-start)/60:.0f} min | running {{{', '.join(f'{v}: {progress(v)}/{SEEDS}' for v in running)}}} | done {sorted(codes)}", flush=True)
print(codes)
for v, c in codes.items():
    if c != 0:
        print(f"--- {v} FAILED (code {c}); last lines of its log:")
        print("".join(open(f"{WORK}/results/log_r4_{v}.txt").readlines()[-25:]))

In [ ]:
out = subprocess.run([sys.executable, "-W", "ignore", "-m", "benchmark.report4"], cwd=WORK, capture_output=True, text=True)
print(out.stdout[-9000:]); print(out.stderr[-2000:])
open(f"{WORK}/results/report4.txt", "w").write(out.stdout)

In [ ]:
with zipfile.ZipFile("/kaggle/working/round4_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(WORK + "/results/*"):
        if os.path.basename(f).startswith(("preds_r4_", "info_r4_", "log_r4_", "report4")):
            z.write(f, os.path.basename(f))
print("wrote /kaggle/working/round4_results.zip")